# Testing Notebook for EdGB Fisher Analysis

## Goal

This notebook tests the PhenomXHM waveform with a -1PN EdGB phase correction. The model used below is `PhenomXHM_EdGB()`, whose extra Fisher parameter is `alpha_EdGB_km_2 = alpha_EdGB^2` in km$^4$. The relative inclusion in the original GWJulia can be found in the $\texttt{.jl}$ file with the same name. 

## Current code organization

* `PhenomXHM_EdGB.jl` contains the EdGB-specific mapping from `alpha_EdGB_km_2` to the generic TIGER/ppE coefficient `delta_phi_minus2` (for a -1PN addition). The expression for the correction is given by: 

$$\delta \phi_{-2} = \frac{128.0}{3.0} \ \beta_\mathrm{EdGB} \ \eta^{(-2 / 5)}$$
 
where $\beta_\mathrm{EdGB}$ is the expression found in Eq. (4) of [arXiv:1905.00870v3](https://arxiv.org/abs/1905.00870v3).

* `PhenomXHM_TIGER_spinless(-1.0)` (-1.0 indicates the correction) is available as the generic model where the extra parameter is directly `delta_phi_minus2` and is the one called in the methods to return the polarizations hphc. 

* `PhenomXHM.jl` and `ConnectionFunctionsXAS.jl` only carry the generic -1PN phase plumbing, through the `PhenomXHM_TIGER_spinless(-1.0)` structure.


In [8]:
using Pkg
Pkg.activate(".")
Pkg.instantiate()

  Activating project at `~/GWInference.jl_nicole`


In [9]:
using GWInference
using LinearAlgebra
using DSP
using Statistics

In [10]:
# GW170608-like, strongest/cleanest EdGB test (from paper at least)
m1 = 11.0
m2 = 7.6
mc = (m1*m2)^(3/5) / (m1 + m2)^(1/5) # ANDREA: note that GWJulia works with the redshifted chirp mass (needs a 1+z term)
eta = m1*m2 / (m1 + m2)^2

chi_eff = 0.03
chi1 = chi_eff * (m1 + m2) / m1
chi2 = 0.0

dL = 0.32  # Gpc
squared_alpha_EdGB_km4 = 2.0^4 # from reference paper cited treshold


theta = 1.0
phi = 2.0
iota = 0.7
psi = 0.4
tcoal = 0.0
phiCoal = 0.0;

In [11]:
fcut = _fcut(PhenomXHM(), mc, eta)

2183.069285129059

## Test 1

Test whether the waveform with $o1=0.0$ is the same as the one without passing the BGR factor.

In [22]:
fmin = 10.
fcut = _fcut(PhenomXHM(), mc, eta, chi1, chi2)
f = collect(range(fmin, fcut, length=2000))


# std GR model
hp_gr, hc_gr = hphc(PhenomXHM(), f, mc, eta, chi1, chi2, dL, iota)

# BGR model with sqrt_alpha_EdGB_km = 0.0 --> means zero added effect in o1 parameter
hp_0, hc_0 = hphc(
    PhenomXHM_EdGB(),
    f, mc, eta, chi1, chi2, dL, iota, 0.0
)

println("GR vs BGR(alpha_EdGB_km^2=0): ", maximum(abs.(hp_gr .- hp_0)))

GR vs BGR(alpha_EdGB_km^2=0): 0.0


## Test 2

Test whether the waveform with a nonzero EdGB coupling is different from GR and finite. The diagnostic value is intentionally not tiny, otherwise the difference is below floating-point precision.

In [38]:
# BGR model with sqrt_alpha_EdGB_km = 0.3
alpha_EdGB_km_2 = 0.3
hp_bgr, hc_bgr = hphc(
    PhenomXHM_EdGB(),
    f, mc, eta, chi1, chi2, dL, iota, 0.3
)

println("GR vs BGR(alpha_EdGB_km^2=0.3): ", maximum(abs.(hp_gr .- hp_bgr)))
println("finite? ", all(isfinite, real.(hp_bgr)), " ", all(isfinite, imag.(hp_bgr)))

GR vs BGR(alpha_EdGB_km^2=0.3): 1.8358176266136343e-23
finite? true true


## Test 2b (ANDREA)
Check that the derivative of the waveform exist and is finite

In [24]:
using ForwardDiff

It takes ages, no idea why, uncomment only if you have ten minutes to spare. The test is passed

In [ ]:
# # BGR model with sqrt_alpha_EdGB_km = 0.3
# params = [mc, eta, chi1, chi2, dL, iota, 0.3]

# hp_bgr_jac_real = ForwardDiff.jacobian(
#     p -> real.(hphc(PhenomXHM_EdGB(), f, p...)[1]),
#     params
# )

# hc_bgr_jac_real = ForwardDiff.jacobian(
#     p -> real.(hphc(PhenomXHM_EdGB(), f, p...)[2]),
#     params
# )

# hp_bgr_jac_imag = ForwardDiff.jacobian(
#     p -> imag.(hphc(PhenomXHM_EdGB(), f, p...)[1]),
#     params
# )

# hc_bgr_jac_imag = ForwardDiff.jacobian(
#     p -> imag.(hphc(PhenomXHM_EdGB(), f, p...)[2]),
#     params
# )

# hp_bgr_jac = hp_bgr_jac_real + im * hp_bgr_jac_imag
# hc_bgr_jac = hc_bgr_jac_real + im * hc_bgr_jac_imag



# println("GR vs BGR(alpha_EdGB_km^2=0.3): ", maximum(abs.(hp_gr .- hp_bgr)))
# println("finite? ", all(isfinite, real.(hp_bgr)), " ", all(isfinite, imag.(hp_bgr)))

# println("finite? ", all(isfinite, real.(hc_bgr_jac)), " ", all(isfinite, imag.(hc_bgr_jac)))



GR vs BGR(alpha_EdGB_km^2=0.3): 1.8358176266136343e-23
finite? true true


## Test 3

Check the expected EdGB scaling. Since the parameter is simple the square of `alpha_EdGB_km`, namely `alpha_EdGB_km_2`, the phase correction scales approximately as `alpha_EdGB_km_2^2`, so doubling the parameter should give a ratio close to 2 for small values:

In [14]:
hp_a, _ = hphc(PhenomXHM_EdGB(), f, mc, eta, chi1, chi2, dL, iota, 0.3)
hp_b, _ = hphc(PhenomXHM_EdGB(), f, mc, eta, chi1, chi2, dL, iota, 0.6)

dh_a = hp_a .- hp_gr
dh_b = hp_b .- hp_gr

println("quadratic scaling ratio: ", norm(dh_b) / norm(dh_a))

quadratic scaling ratio: 1.929489399614246


## Test 4: Fisher analysis

Checks Fisher on waveforms other than mine, and on mine. Then calculate the SNR and covariance to extract errors. 

In [ ]:
FisherMatrix(
    PhenomD(),
    CE1Id,
    mc, eta, chi1, chi2, dL,
    theta, phi, iota, psi, tcoal, phiCoal;
    # res=100,
    # fmin=10.0,
    # fmax=512.0,
    rho_thres=nothing
);

println("Fisher matrix calculated")

Fisher matrix calculated


In [ ]:
model = PhenomXHM_EdGB()

# find the Fisher matrix for the BGR model with sqrt_alpha_EdGB_km = 0.3

F= FisherMatrix(
    model,
    ETS,
    mc,
    eta,
    chi1,
    chi2,
    dL,
    theta,
    phi,
    iota,
    psi,
    tcoal,
    phiCoal,
    alpha_EdGB_km_2;
    # res=300,
    # fmin=10.0,
    # fmax=512.0,
    rho_thres=nothing,
    #return_SNR=true
);

println("Fisher matrix calculated")

UndefVarError: UndefVarError: `alpha_EdGB_km_2` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
println(size(F))
println("is symmetric? (0 means true) = ", maximum(abs.(F .- F')) ) # ANDREA: true by design in gwjulia
println("is finite? = ", all(isfinite, F))
println("diag = ", diag(F))

UndefVarError: UndefVarError: `F` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.
Hint: a global variable of this name also exists in Elliptic.

In [ ]:
snr = SNR(
    model,
    ETS,
    mc, eta, chi1, chi2, dL,
    theta, phi, iota, psi, tcoal,
    alpha_EdGB_km_2;
    # res=100,
    # fmin=10.0,
    # fmax=512.0
)

println("signal SNR = ", snr) # ANDREA: one can check that the snr does not change in the two cases (GR vs BGR) since the SNR is calculated with the amplitude (which the BGR does not affect)

UndefVarError: UndefVarError: `alpha_EdGB_km_2` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [19]:
mycovariance = CovMatrix(F);

UndefVarError: UndefVarError: `F` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.
Hint: a global variable of this name also exists in Elliptic.

In [20]:
# we can now calculate the errors on the parameters

myerrors = Errors(mycovariance)
parameters_string = ["mc", "η", "χ_1", "χ_2", "dL", "θ", "ϕ", "ι", "ψ", "tcoal", "Φ_coal", "α_EdGB^2 [km^4]"]

for i in eachindex(myerrors)
    println("The error on $(parameters_string[i]) is $(myerrors[i])")
end

UndefVarError: UndefVarError: `mycovariance` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.

In [21]:
# find errors on alpha_EdGB, which is the param in the paper we are interested in:

sigma_alpha_squared = myerrors[end]
sigma_sqrt_alpha = sigma_alpha_squared^(1.0 / 4.0)

println("The error on sqrt(α_EdGB^2) is $(sigma_sqrt_alpha)")

UndefVarError: UndefVarError: `myerrors` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.

It's important to notice that I am studying the Fisher matrix for the squared value of the parameters alpha in a way that the derivative with respect to such parameter will be linear and not singular for small values of such parameter. 